# 🤖 Lección 5 – Machine Learning Escalable con Spark MLlib
### Proyecto: Retail Analytics Pipeline — RetailMax
**Módulo 9: Fundamentos de Big Data | Alkemy**

---
**Objetivo:** Construir un pipeline completo de MLlib para clasificación supervisada (Regresión Logística) y segmentación no supervisada (K-Means) del catálogo de productos Fashion-MNIST.

In [ ]:
# ============================================================
# Setup
# ============================================================
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

# MLlib — Feature Engineering
from pyspark.ml.feature import (
    VectorAssembler, StringIndexer, StandardScaler,
    PCA, IndexToString
)
# MLlib — Modelos
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.clustering import KMeans
# MLlib — Pipeline y Evaluación
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import (
    MulticlassClassificationEvaluator,
    ClusteringEvaluator
)
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

import os, numpy as np, matplotlib.pyplot as plt, matplotlib
import seaborn as sns, pandas as pd

spark = (
    SparkSession.builder
    .appName('RetailMax_L5_MLlib')
    .master('local[*]')
    .config('spark.driver.memory', '3g')
    .config('spark.sql.shuffle.partitions', '8')
    .config('spark.ui.showConsoleProgress', 'false')
    .getOrCreate()
)
sc = spark.sparkContext
sc.setLogLevel('ERROR')
print(f'✅ SparkSession lista | Spark {sc.version}')

In [ ]:
# ============================================================
# 1. Cargar datos desde Parquet (generado en Lección 4)
# ============================================================
OUTPUTS = '../outputs'
DATA_PATH = '../data/fashion_mnist'

# Intentar cargar desde Parquet; si no existe, regenerar
metrics_parquet = os.path.join(OUTPUTS, 'fashion_metrics.parquet')

if os.path.exists(metrics_parquet):
    df_metrics = spark.read.parquet(metrics_parquet)
    print(f'✅ Parquet cargado desde Lección 4')
else:
    print('⚠️  Parquet no encontrado. Cargando desde CSV...')
    pixel_fields = [StructField(f'pixel_{i}', IntegerType(), True) for i in range(784)]
    schema = StructType([StructField('label', IntegerType(), False),
                         StructField('label_name', StringType(), False)] + pixel_fields)
    df_raw = spark.read.option('header','true').schema(schema)\
                  .csv(os.path.join(DATA_PATH, 'fashion_train.csv'))
    pixel_cols = [f'pixel_{i}' for i in range(784)]
    df_metrics = df_raw.withColumn('intensidad_media', sum([F.col(p) for p in pixel_cols])/784)\
                       .withColumn('pixel_max', F.greatest(*[F.col(p) for p in pixel_cols]))\
                       .withColumn('pixel_min', F.least(*[F.col(p) for p in pixel_cols]))\
                       .withColumn('contraste', F.col('pixel_max') - F.col('pixel_min'))\
                       .withColumn('segmento_brillo',
                                   F.when(F.col('intensidad_media')<64, 'oscuro')\
                                    .when(F.col('intensidad_media')<128,'medio')\
                                    .otherwise('claro'))\
                       .select('label','label_name','intensidad_media',
                               'pixel_max','pixel_min','contraste','segmento_brillo')

df_metrics.cache()
print(f'Rows: {df_metrics.count():,} | Cols: {df_metrics.columns}')
df_metrics.show(3)

## 2. Feature Engineering para MLlib

MLlib requiere que las features estén en una **columna de tipo Vector**. Los pasos son:

1. `StringIndexer` → convierte `label_name` (string) en índice numérico
2. `VectorAssembler` → combina columnas numéricas en un vector de features
3. `StandardScaler` → normaliza el vector (media 0, desviación 1)

In [ ]:
# ============================================================
# 2. StringIndexer: label_name → índice numérico
# ============================================================
indexer = StringIndexer(
    inputCol='label_name',
    outputCol='label_index',
    handleInvalid='keep'
)
df_indexed = indexer.fit(df_metrics).transform(df_metrics)

print('=== StringIndexer: label_name → label_index ===')
df_indexed.select('label_name', 'label_index').distinct().orderBy('label_index').show()

# Guardar mapping para interpretación posterior
label_mapping = {
    row['label_index']: row['label_name'] 
    for row in df_indexed.select('label_name','label_index').distinct().collect()
}

In [ ]:
# ============================================================
# 3. VectorAssembler: features numéricas → vector
# ============================================================
FEATURE_COLS = ['intensidad_media', 'pixel_max', 'pixel_min', 'contraste']

assembler = VectorAssembler(
    inputCols=FEATURE_COLS,
    outputCol='features_raw'
)

# StandardScaler: normalizar features
scaler = StandardScaler(
    inputCol='features_raw',
    outputCol='features',
    withStd=True,
    withMean=True
)

print(f'Features seleccionadas: {FEATURE_COLS}')
print('VectorAssembler y StandardScaler configurados.')

## 3. Pipeline Supervisado — Regresión Logística

Clasificar imágenes de prendas en sus 10 categorías usando las métricas visuales como features.

In [ ]:
# ============================================================
# 4. Split train/test y pipeline de Regresión Logística
# ============================================================
# Preparar datos con el label correcto
df_ml = df_indexed.select(
    'label_index',
    'label_name',
    'intensidad_media',
    'pixel_max',
    'pixel_min',
    'contraste'
)

# Split 80/20
df_train_ml, df_test_ml = df_ml.randomSplit([0.8, 0.2], seed=42)
df_train_ml.cache()
df_test_ml.cache()

print(f'Train: {df_train_ml.count():,} | Test: {df_test_ml.count():,}')

# Modelo de Regresión Logística Multinomial
lr = LogisticRegression(
    featuresCol='features',
    labelCol='label_index',
    maxIter=30,
    regParam=0.01,
    elasticNetParam=0.0,
    family='multinomial'
)

# Pipeline completo: indexer → assembler → scaler → lr
pipeline_lr = Pipeline(stages=[
    indexer,
    assembler,
    scaler,
    lr
])

print('Pipeline de Regresión Logística configurado.')

In [ ]:
# ============================================================
# 5. Entrenar el modelo
# ============================================================
print('Entrenando modelo de Regresión Logística...')

# El pipeline recibe datos originales y aplica todas las etapas
df_train_input = df_indexed.select(
    'label_name', 'label_index',
    'intensidad_media', 'pixel_max', 'pixel_min', 'contraste'
)
df_train_subset, df_test_subset = df_train_input.randomSplit([0.8, 0.2], seed=42)

# Fit: ajusta todos los stages del pipeline
model_lr = pipeline_lr.fit(df_train_subset)

# Predict sobre test
predictions_lr = model_lr.transform(df_test_subset)

print('✅ Modelo entrenado exitosamente.')
predictions_lr.select('label_name', 'label_index', 'prediction', 'probability').show(5)

In [ ]:
# ============================================================
# 6. Evaluación — Métricas de clasificación
# ============================================================
evaluator_acc  = MulticlassClassificationEvaluator(
    labelCol='label_index', predictionCol='prediction', metricName='accuracy')
evaluator_f1   = MulticlassClassificationEvaluator(
    labelCol='label_index', predictionCol='prediction', metricName='f1')
evaluator_prec = MulticlassClassificationEvaluator(
    labelCol='label_index', predictionCol='prediction', metricName='weightedPrecision')
evaluator_rec  = MulticlassClassificationEvaluator(
    labelCol='label_index', predictionCol='prediction', metricName='weightedRecall')

acc  = evaluator_acc.evaluate(predictions_lr)
f1   = evaluator_f1.evaluate(predictions_lr)
prec = evaluator_prec.evaluate(predictions_lr)
rec  = evaluator_rec.evaluate(predictions_lr)

print('=== MÉTRICAS — REGRESIÓN LOGÍSTICA ===')
print(f'  Accuracy            : {acc:.4f}  ({acc*100:.2f}%)')
print(f'  F1-Score (weighted) : {f1:.4f}')
print(f'  Precision (weighted): {prec:.4f}')
print(f'  Recall (weighted)   : {rec:.4f}')
print()
print('💡 Nota: Con solo 4 features (intensidad, contraste, min, max)')
print('   se espera una accuracy entre 15-30%. Un modelo completo con')
print('   los 784 píxeles como features puede superar el 85%.')

In [ ]:
# ============================================================
# Matriz de confusión por categoría
# ============================================================
# Calcular accuracy por clase
acc_por_clase = (
    predictions_lr
    .groupBy('label_name', 'label_index', 'prediction')
    .count()
    .orderBy('label_index', 'prediction')
)

# Accuracy correctas por clase
acc_clase = (
    predictions_lr
    .withColumn('correct', (F.col('label_index') == F.col('prediction')).cast('int'))
    .groupBy('label_name')
    .agg(
        F.sum('correct').alias('correctas'),
        F.count('*').alias('total')
    )
    .withColumn('accuracy_clase', F.round(F.col('correctas')/F.col('total'), 4))
    .orderBy('accuracy_clase', ascending=False)
)

print('=== ACCURACY POR CLASE ===')
acc_clase.show(truncate=False)
df_acc_pd = acc_clase.toPandas()

## 4. Pipeline No Supervisado — K-Means

Segmentar el catálogo de productos en grupos naturales sin usar las etiquetas, para descubrir patrones visuales y generar segmentos para marketing.

In [ ]:
# ============================================================
# 7. Pipeline K-Means para segmentación
# ============================================================
# Preparar features para clustering (sin usar label)
assembler_km = VectorAssembler(
    inputCols=FEATURE_COLS,
    outputCol='features_raw_km'
)
scaler_km = StandardScaler(
    inputCol='features_raw_km',
    outputCol='features_km',
    withStd=True, withMean=True
)

# K-Means con k=5 (5 segmentos de marketing)
K = 5
kmeans = KMeans(
    featuresCol='features_km',
    predictionCol='cluster',
    k=K,
    seed=42,
    maxIter=20
)

pipeline_km = Pipeline(stages=[assembler_km, scaler_km, kmeans])

df_km_input = df_indexed.select('label_name', *FEATURE_COLS)

print(f'Entrenando K-Means con k={K}...')
model_km = pipeline_km.fit(df_km_input)
predictions_km = model_km.transform(df_km_input)
print('✅ K-Means entrenado.')

In [ ]:
# ============================================================
# 8. Evaluación K-Means — Silhouette Score
# ============================================================
evaluator_km = ClusteringEvaluator(
    featuresCol='features_km',
    predictionCol='cluster',
    metricName='silhouette'
)
silhouette = evaluator_km.evaluate(predictions_km)

print(f'=== MÉTRICAS K-MEANS (k={K}) ===')
print(f'  Silhouette Score: {silhouette:.4f}')
print(f'  (Rango: [-1, 1] | >0.5 = buena separación | <0.2 = clusters solapados)')

# Distribución de categorías por cluster
print('\n=== COMPOSICIÓN DE CLUSTERS ===')
cluster_comp = (
    predictions_km
    .groupBy('cluster', 'label_name')
    .count()
    .orderBy('cluster', 'count', ascending=[True, False])
)
cluster_comp.show(50, truncate=False)

In [ ]:
# ============================================================
# 9. Determinar el K óptimo (Elbow Method)
# ============================================================
print('Calculando WCSS para k=2..8 (Elbow Method)...')

# Preparar features (solo una vez)
df_elbow = assembler_km.transform(df_km_input)
df_elbow = scaler_km.fit(df_elbow).transform(df_elbow)

wcss_values = []
k_range = range(2, 9)

for k in k_range:
    km_tmp = KMeans(featuresCol='features_km', predictionCol='cluster',
                    k=k, seed=42, maxIter=15)
    model_tmp = km_tmp.fit(df_elbow)
    wcss = model_tmp.summary.trainingCost
    wcss_values.append(wcss)
    print(f'  k={k}: WCSS = {wcss:.2f}')

# Gráfico del codo
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(list(k_range), wcss_values, 'bo-', linewidth=2, markersize=8)
ax.axvline(x=K, color='red', linestyle='--', label=f'k={K} seleccionado')
ax.set_xlabel('Número de clusters (k)', fontsize=12)
ax.set_ylabel('Within-Cluster Sum of Squares (WCSS)', fontsize=12)
ax.set_title('Elbow Method — K-Means\nRetailMax: Segmentación de Productos', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

os.makedirs('../visualizations', exist_ok=True)
plt.savefig('../visualizations/L5_elbow_kmeans.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Gráfico guardado: visualizations/L5_elbow_kmeans.png')

In [ ]:
# ============================================================
# 10. Visualizaciones finales del pipeline ML
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('RetailMax — Resultados MLlib Pipeline\nRegresión Logística + K-Means',
             fontsize=14, fontweight='bold')

# Izquierda: Accuracy por clase (Logistic Regression)
df_acc_sorted = df_acc_pd.sort_values('accuracy_clase')
colors_acc = ['#d32f2f' if v < 0.15 else '#f57c00' if v < 0.25 else '#388e3c' 
              for v in df_acc_sorted['accuracy_clase']]
bars = axes[0].barh(df_acc_sorted['label_name'], df_acc_sorted['accuracy_clase'] * 100, 
                    color=colors_acc)
axes[0].set_xlabel('Accuracy (%)')
axes[0].set_title('Accuracy por Clase\n(Regresión Logística)', fontweight='bold')
axes[0].axvline(x=acc*100, color='blue', linestyle='--', label=f'Media: {acc*100:.1f}%')
axes[0].legend()
for bar, val in zip(bars, df_acc_sorted['accuracy_clase']):
    axes[0].text(val*100 + 0.3, bar.get_y() + bar.get_height()/2,
                 f'{val*100:.1f}%', va='center', fontsize=8)

# Derecha: Distribución de clusters K-Means
cluster_dist_pd = (
    predictions_km
    .groupBy('cluster')
    .count()
    .orderBy('cluster')
    .toPandas()
)
cluster_labels = [f'Segmento {i}' for i in cluster_dist_pd['cluster']]
axes[1].pie(cluster_dist_pd['count'], labels=cluster_labels, autopct='%1.1f%%',
            colors=plt.cm.Set3(np.linspace(0, 1, K)), startangle=90)
axes[1].set_title(f'Distribución de Segmentos K-Means (k={K})\n'
                  f'Silhouette: {silhouette:.3f}', fontweight='bold')

plt.tight_layout()
plt.savefig('../visualizations/L5_resultados_ml.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Visualización guardada: visualizations/L5_resultados_ml.png')

In [ ]:
# ============================================================
# 11. Descripción de segmentos para Marketing
# ============================================================
print('\n' + '='*60)
print('REPORTE DE SEGMENTOS PARA MARKETING — RetailMax')
print('='*60)

segmentos = (
    predictions_km
    .groupBy('cluster')
    .agg(
        F.count('*').alias('total_productos'),
        F.round(F.avg('intensidad_media'), 2).alias('brillo_prom'),
        F.round(F.avg('contraste'), 2).alias('contraste_prom'),
        F.collect_set('label_name').alias('categorias')
    )
    .orderBy('cluster')
)

for row in segmentos.collect():
    cats = ', '.join(sorted(row['categorias']))
    print(f'\nSegmento {row["cluster"]} ({row["total_productos"]:,} productos):')
    print(f'  Brillo promedio   : {row["brillo_prom"]}')
    print(f'  Contraste promedio: {row["contraste_prom"]}')
    print(f'  Categorías        : {cats}')
    # Interpretación de marketing
    if row['brillo_prom'] > 100:
        tipo = '🌟 Productos de alta visibilidad — ideal para destacar en homepage'
    elif row['contraste_prom'] > 150:
        tipo = '🎨 Productos con fuerte contraste visual — buenas miniaturas para catálogo'
    else:
        tipo = '🔲 Productos sutiles — requieren descripción detallada en fichas'
    print(f'  Insight marketing : {tipo}')

print('\n' + '='*60)

In [ ]:
# ============================================================
# 12. Guardar predicciones y modelo
# ============================================================
# Guardar predicciones K-Means como CSV para el informe
pred_csv = os.path.join(OUTPUTS, 'predicciones_kmeans.csv')
predictions_km.select('label_name','contraste','intensidad_media','cluster')\
               .toPandas().to_csv(pred_csv, index=False)

pred_lr_csv = os.path.join(OUTPUTS, 'predicciones_lr.csv')
predictions_lr.select('label_name','label_index','prediction')\
               .toPandas().to_csv(pred_lr_csv, index=False)

# Guardar modelo LR
model_path = os.path.join(OUTPUTS, 'model_lr_pipeline')
model_lr.write().overwrite().save(model_path)

model_km_path = os.path.join(OUTPUTS, 'model_kmeans_pipeline')
model_km.write().overwrite().save(model_km_path)

print('✅ Artefactos guardados:')
print(f'   {pred_csv}')
print(f'   {pred_lr_csv}')
print(f'   {model_path}')
print(f'   {model_km_path}')

---
## ✅ Checklist Lección 5
- [x] DataFrames cargados desde Parquet (Lección 4)
- [x] `StringIndexer` aplicado para convertir labels a índices
- [x] `VectorAssembler` + `StandardScaler` para feature engineering
- [x] Pipeline supervisado: **Regresión Logística Multinomial** entrenada y evaluada
- [x] Métricas: Accuracy, F1, Precision, Recall por clase
- [x] Pipeline no supervisado: **K-Means** con k=5 y evaluación Silhouette
- [x] Elbow Method para selección de k óptimo
- [x] Segmentos de marketing interpretados
- [x] Modelos guardados en disco
- [x] Visualizaciones generadas

---
### 🎯 Pipeline completo implementado:
```
CSV (Fashion-MNIST)
  └─ Spark RDD (Lección 3)
      └─ DataFrame + Spark SQL (Lección 4)
          └─ Parquet
              └─ MLlib Pipeline
                  ├─ StringIndexer
                  ├─ VectorAssembler
                  ├─ StandardScaler
                  ├─ LogisticRegression → Clasificación de categorías
                  └─ KMeans            → Segmentación para marketing
```